# Pipeline Engineering Practices

Correct logic is not enough for production pipelines. They must also be **observable**, **resilient**, and **safe to re-run**. This notebook covers four foundational practices every data engineer must apply:

| Practice | Problem it solves |
|---|---|
| **Configuration Management** | Hard-coded values break across environments |
| **Structured Logging** | Plain text logs are hard to search and alert on |
| **Retry with Exponential Backoff** | Transient failures kill brittle pipelines |
| **Idempotency** | Re-running a failed pipeline duplicates data |

In [ ]:
import os
import re
import json
import sys
import time
import random
import hashlib
import logging
import functools
import yaml
from pathlib import Path

print('Dependencies loaded')

## 1. Configuration Management

Config must be **external to code** so the same artifact runs in dev, staging, and production without changes.

- **YAML files** — human-readable, version-controlled, shared across the team
- **Environment variables** — secrets and per-environment overrides (12-factor app)
- **${VAR} placeholders** — defer secret resolution to runtime

Never commit secrets (passwords, API keys) to version control.

In [ ]:
# Build the config as a Python dict, then serialize to YAML
# ${...} placeholders are resolved at runtime from environment variables
config_data = {
    'pipeline': {
        'name': 'sales_etl',
        'version': '1.0',
        'environment': 'development',
    },
    'source': {
        'type': 'mysql',
        'host': '${DB_HOST}',
        'port': 3306,
        'database': 'classicmodels',
        'table': 'orders',
        'password': '${DB_PASSWORD}',
    },
    'destination': {
        'type': 's3',
        'bucket': 'my-data-lake',
        'prefix': 'raw/orders/',
    },
    'settings': {
        'batch_size': 1000,
        'retry_attempts': 3,
        'retry_delay_seconds': 2,
        'timeout_seconds': 30,
    },
}

config_path = Path('pipeline_config.yaml')
with open(config_path, 'w') as f:
    yaml.dump(config_data, f, default_flow_style=False, sort_keys=False)

print(config_path.read_text())

In [ ]:
def load_config(path):
    '''Load YAML config; resolve ${VAR} placeholders from environment variables.'''
    with open(path) as f:
        content = f.read()
    def _resolve(match):
        var = match.group(1)
        return os.environ.get(var, f'<{var}_NOT_SET>')
    content = re.sub(r'\$\{(\w+)\}', _resolve, content)
    return yaml.safe_load(content)

cfg = load_config(config_path)

name    = cfg['pipeline']['name']
version = cfg['pipeline']['version']
db      = cfg['source']['database']
table   = cfg['source']['table']
batch   = cfg['settings']['batch_size']

print(f'Pipeline : {name} v{version}')
print(f'Source   : {db}.{table}')
print(f'Batch    : {batch} rows/run')
print(f'DB host  : {cfg["source"]["host"]}')   # shows placeholder if env var not set

## 2. Structured Logging

Plain-text logs (`print` or unformatted `logging`) are hard to:
- Search across thousands of lines
- Forward to observability platforms (Datadog, CloudWatch, ELK)
- Set metric-based alerts on

**Structured logs** emit JSON — every field is queryable and filterable.

In [ ]:
class StructuredFormatter(logging.Formatter):
    '''Emits log records as single-line JSON for ingestion by log aggregators.'''

    def format(self, record):
        entry = {
            'level':   record.levelname,
            'logger':  record.name,
            'message': record.getMessage(),
            'module':  record.module,
        }
        # Caller can attach extra fields by prefixing kwarg names with 'ctx_'
        entry.update(
            {k[4:]: v for k, v in record.__dict__.items() if k.startswith('ctx_')}
        )
        if record.exc_info:
            entry['exception'] = self.formatException(record.exc_info)
        return json.dumps(entry)


def get_logger(name, level=logging.INFO):
    logger = logging.getLogger(name)
    logger.setLevel(level)
    if not logger.handlers:
        h = logging.StreamHandler(sys.stdout)
        h.setFormatter(StructuredFormatter())
        logger.addHandler(h)
    return logger


log = get_logger('sales_etl')
log.info('Pipeline started')
log.info('Batch ready', extra={'ctx_batch_id': 'b001', 'ctx_rows': 1000, 'ctx_source': 'orders'})
log.warning('Slow query detected', extra={'ctx_duration_ms': 4800, 'ctx_table': 'orders'})

## 3. Custom Pipeline Exceptions

Generic `Exception` tells operators nothing about what went wrong or where.
A typed exception hierarchy enables:
- Targeted `except` clauses (catch extraction errors differently from load errors)
- Structured error messages with context attached to the exception object
- Clear monitoring alerts ("LoadError spike in prod")

In [ ]:
class PipelineError(Exception):
    '''Base class for all pipeline errors.'''

class ExtractionError(PipelineError):
    def __init__(self, source, reason):
        self.source = source
        super().__init__(f'Extraction from {source!r} failed: {reason}')

class TransformError(PipelineError):
    def __init__(self, step, reason):
        self.step = step
        super().__init__(f'Transform step {step!r} failed: {reason}')

class LoadError(PipelineError):
    def __init__(self, destination, reason):
        self.destination = destination
        super().__init__(f'Load to {destination!r} failed: {reason}')


def validate_record(record):
    '''Raise TransformError for records that violate data contracts.'''
    if 'order_number' not in record:
        raise TransformError('validate', 'missing required field: order_number')
    amount = record.get('amount', 0)
    if amount < 0:
        raise TransformError('validate', f'negative amount: {amount}')
    return record


# Demo: structured exception capture
bad_records = [
    {'customer': 'C1', 'amount': -50.0},          # missing order_number
    {'order_number': 1002, 'amount': -9.99},       # negative amount
    {'order_number': 1003, 'amount': 350.0},       # valid
]
for r in bad_records:
    try:
        validate_record(r)
        log.info('Record valid', extra={'ctx_order': r.get('order_number')})
    except TransformError as e:
        log.error(str(e), extra={'ctx_step': e.step, 'ctx_record': r})

## 4. Retry with Exponential Backoff

Network timeouts, throttling errors, and brief database unavailability are **transient** — a retry a few seconds later will often succeed.

**Exponential backoff** grows the wait time after each failure to avoid overwhelming the service you're retrying against. **Jitter** randomizes the delay so retries from many parallel workers don't all hit the service at the same moment (thundering-herd problem).

In [ ]:
def retry(
    exceptions=(Exception,),
    max_attempts=3,
    base_delay=1.0,
    backoff_factor=2.0,
    max_delay=30.0,
    jitter=True,
):
    '''Decorator: retry a function on specified exceptions with exponential backoff.'''
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            delay = base_delay
            for attempt in range(1, max_attempts + 1):
                try:
                    return fn(*args, **kwargs)
                except exceptions as exc:
                    if attempt == max_attempts:
                        log.error(f'{fn.__name__} failed after {max_attempts} attempts: {exc}')
                        raise
                    sleep = min(delay, max_delay)
                    if jitter:
                        sleep *= 0.5 + random.random()   # ±50 % jitter
                    log.warning(
                        f'{fn.__name__} attempt {attempt}/{max_attempts} failed — retry in {sleep:.2f}s',
                        extra={'ctx_error': str(exc), 'ctx_attempt': attempt}
                    )
                    time.sleep(sleep)
                    delay = min(delay * backoff_factor, max_delay)
        return wrapper
    return decorator


# Demo: simulate a flaky source connection that succeeds on the 3rd attempt
_attempts = 0

@retry(exceptions=(ConnectionError,), max_attempts=3, base_delay=0.05, jitter=False)
def connect_to_source(host):
    global _attempts
    _attempts += 1
    if _attempts < 3:
        raise ConnectionError(f'connection refused by {host} (attempt {_attempts})')
    return f'connected to {host} on attempt {_attempts}'


result = connect_to_source('db.classicmodels.internal')
print(result)

## 5. Idempotency

A pipeline is **idempotent** if running it multiple times produces the same result as running it once. This is essential because:
- Pipelines fail mid-run and must be safely re-triggered
- Orchestrators may duplicate task execution
- Backfills re-process historical data

**Pattern**: derive a deterministic key from each record's natural identity, track processed keys in a durable store, and skip records whose key has already been seen.

In [ ]:
class IdempotencyStore:
    '''Tracks processed record keys to prevent duplicate loads.
    Backed by a local JSON file; replace with Redis or DynamoDB in production.
    '''

    def __init__(self, store_path='processed_keys.json'):
        self._path = Path(store_path)
        self._seen = set(json.loads(self._path.read_text())) if self._path.exists() else set()

    def record_key(self, record):
        '''Deterministic key derived from the record natural identity.
        Use business keys (order_number, not auto-increment id) for stability.
        '''
        natural_id = {'order_number': record['order_number']}
        payload = json.dumps(natural_id, sort_keys=True).encode()
        return hashlib.sha256(payload).hexdigest()[:16]

    def is_processed(self, key):
        return key in self._seen

    def mark_processed(self, key):
        self._seen.add(key)
        self._path.write_text(json.dumps(sorted(self._seen)))


store = IdempotencyStore()

# Simulate a batch that includes a duplicate (order 10001 appears twice)
orders = [
    {'order_number': 10001, 'amount': 350.0},
    {'order_number': 10002, 'amount': 820.5},
    {'order_number': 10001, 'amount': 350.0},   # duplicate — must be skipped
    {'order_number': 10003, 'amount': 120.0},
]

loaded = skipped = 0
for order in orders:
    num = order['order_number']
    key = store.record_key(order)
    if store.is_processed(key):
        log.info(f'Skipping duplicate (key={key})', extra={'ctx_order': num})
        skipped += 1
    else:
        log.info(f'Loading order {num} (key={key})')
        store.mark_processed(key)
        loaded += 1

print(f'\nResult: {loaded} loaded, {skipped} skipped')

## 6. Putting It Together — A Resilient Pipeline

A production-grade pipeline applies all four practices together:

In [ ]:
class SalesETL:
    '''Resilient ETL pipeline with config management, structured logging,
    retry on extraction, validation, and idempotent loading.
    '''

    def __init__(self, config):
        self.cfg   = config
        self.log   = get_logger(config['pipeline']['name'])
        self.store = IdempotencyStore()

    @retry(exceptions=(ExtractionError,), max_attempts=3, base_delay=0.1, jitter=False)
    def extract(self):
        self.log.info('Extracting from source', extra={'ctx_table': self.cfg['source']['table']})
        # In production: run a SELECT against the source database
        return [
            {'order_number': 20001, 'amount': 350.0},
            {'order_number': 20002, 'amount': -5.0},    # will fail validation
            {'order_number': 20003, 'amount': 820.5},
            {'order_number': 20001, 'amount': 350.0},   # duplicate
        ]

    def transform(self, records):
        valid, dropped = [], 0
        for r in records:
            try:
                valid.append(validate_record(r))
            except TransformError as e:
                dropped += 1
                self.log.warning(str(e), extra={'ctx_step': e.step})
        self.log.info(f'Transform complete', extra={'ctx_valid': len(valid), 'ctx_dropped': dropped})
        return valid

    def load(self, records):
        bucket = self.cfg['destination']['bucket']
        loaded = skipped = 0
        for r in records:
            key = self.store.record_key(r)
            if self.store.is_processed(key):
                skipped += 1
            else:
                self.store.mark_processed(key)
                loaded += 1
        self.log.info('Load complete', extra={'ctx_loaded': loaded, 'ctx_skipped': skipped, 'ctx_dest': bucket})
        return loaded

    def run(self):
        pipeline_name = self.cfg['pipeline']['name']
        self.log.info(f'Run started', extra={'ctx_pipeline': pipeline_name})
        try:
            raw   = self.extract()
            clean = self.transform(raw)
            count = self.load(clean)
            self.log.info('Run complete', extra={'ctx_records_loaded': count})
        except PipelineError as e:
            self.log.error(f'Run failed: {e}')
            raise


pipeline = SalesETL(cfg)
pipeline.run()

# Cleanup temp files created during this notebook
for f in ['pipeline_config.yaml', 'processed_keys.json']:
    Path(f).unlink(missing_ok=True)
print('\nTemp files cleaned up')